# RandomForestClassifier → fail-closed decision model (binary **and** multiclass)

**Two problems, one harness.**

1. **Binary** — `breast_cancer` (all 30 features): decide whether a case is safe to call *benign*
   without human review. `p_positive` is P(malignant).
2. **Multiclass** — `wine` (all 13 features): decide whether a sample is the approved class 0;
   `pred`/`conf` and the conformal prediction **set** carry the uncertainty.

Feature names are sanitized (spaces and `/` → `_`) so they are valid DSL lambda parameters — the model
still uses **all** features.

**The shape of every policy is the same:** stated facts + model facts (`pred`, `p_positive`, `conf`,
`certified`, `set_size`) → `release | ask | deny | abstain`. `certified` means the conformal prediction
set is a singleton at α = 0.10; a non-singleton set cannot be released.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from mjolnir_ml import (ASK, DENY, RELEASE, DecisionModel, DecisionPolicyBase,
                        HazardLedger, dsl, report, rule, certify, verify_certificate)

pd.set_option("display.width", 140)
clean = lambda names: tuple(n.replace(" ", "_").replace("/", "_") for n in names)

## Binary — breast_cancer (all 30 features)

Policy (four statements, all four verdicts represented):

- `deny_out_of_spec` — a **stated-fact** hazard (`worst_area > 1500`): a measurement out of spec is refused.
- `deny_confident_malignant` — a confident risky prediction is refused.
- `release_certified_benign` — release only when the conformal set is a **singleton** at α and the risk is low.
- `review_unsettled` — a non-certified prediction escalates (`ask`).

The oracle for the Ω pair is `safe = (label == 0)` (benign).

In [2]:
data = load_breast_cancer()
FEATURES_B = clean(data.feature_names)
Xtr, Xrest, ytr, yrest = train_test_split(data.data, data.target, test_size=0.5,
                                          stratify=data.target, random_state=0)
Xcal, Xte, ycal, yte = train_test_split(Xrest, yrest, test_size=0.5,
                                        stratify=yrest, random_state=0)
rows = lambda A: [dict(zip(FEATURES_B, map(float, x))) for x in A]
tr_b, cal_b, te_b = rows(Xtr), rows(Xcal), rows(Xte)

forest_b = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(Xtr, ytr)
print("features (all):", FEATURES_B)

@dsl
class MammographyPolicy(DecisionPolicyBase):
    assert rule("deny_out_of_spec",
                when=lambda worst_area: worst_area > 1500.0, verdict=DENY)
    assert rule("deny_confident_malignant",
                when=lambda p_positive, conf: p_positive >= 0.80 and conf >= 0.80, verdict=DENY)
    assert rule("release_certified_benign",
                when=lambda certified, p_positive: certified and p_positive < 0.50, verdict=RELEASE)
    assert rule("review_unsettled",
                when=lambda certified: not certified, verdict=ASK)

dm_b = (DecisionModel(forest_b, MammographyPolicy, features=FEATURES_B)
        .fit_regime(tr_b, method="mahalanobis",
                    alpha_ood=0.02, calibration_rows=cal_b).calibrate(cal_b, ycal, alpha=0.10))

ledger_b = (HazardLedger()
            .add("out_of_spec_area", lambda r: r["worst_area"] > 1500.0,
                 "spec sheet §4.1"))
print(report(dm_b, te_b, title="Breast cancer — release benign without review",
             safe=[int(v) == 0 for v in yte], decidable=[True] * len(yte),
             ledger=ledger_b, key=b"demo-audit-key", sample=8))

features (all): ('mean_radius', 'mean_texture', 'mean_perimeter', 'mean_area', 'mean_smoothness', 'mean_compactness', 'mean_concavity', 'mean_concave_points', 'mean_symmetry', 'mean_fractal_dimension', 'radius_error', 'texture_error', 'perimeter_error', 'area_error', 'smoothness_error', 'compactness_error', 'concavity_error', 'concave_points_error', 'symmetry_error', 'fractal_dimension_error', 'worst_radius', 'worst_texture', 'worst_perimeter', 'worst_area', 'worst_smoothness', 'worst_compactness', 'worst_concavity', 'worst_concave_points', 'worst_symmetry', 'worst_fractal_dimension')


 Breast cancer — release benign without review
 mjolnir.ml.report/1 · 2026-10-25T17:39:34Z

CONTRACT (the problem)
----------------------
  policy        MammographyPolicy  (sha256:8827e9656cd34828a358dc20affc760ba36b20b79b6add45d2068f7348727313)
  model         RandomForestClassifier  (sha256:03ae4b5b6409c5b0…)
  calibration   alpha=0.10  (sha256:9627abc1181d023e…)
  regime        required  method=mahalanobis  alpha_ood=0.02  tau=8241.46  (sha256:c367cc692d08c9d8…)
  features      mean_radius, mean_texture, mean_perimeter, mean_area, mean_smoothness, mean_compactness, mean_concavity, mean_concave_points, mean_symmetry, mean_fractal_dimension, radius_error, texture_error, perimeter_error, area_error, smoothness_error, compactness_error, concavity_error, concave_points_error, symmetry_error, fractal_dimension_error, worst_radius, worst_texture, worst_perimeter, worst_area, worst_smoothness, worst_compactness, worst_concavity, worst_concave_points, worst_symmetry, worst_fractal_dimension

### The second guarantee: selective risk (CRC)

`certified` answers a **coverage** question: is the conformal set a singleton for this case? That is a
per-row claim. Operations often need a different one: *among the cases I auto-release, at most α are
malignant*. That is **selective risk**, and it is calibrated by Conformal Risk Control on the same
`p_positive`:

```
release_ok  p_positive <= lambda_release
deny_ok     p_positive >= lambda_deny
review      otherwise
```

The guarantee is on the expected risk of the selected cohort, not on each row's set. This is the
classifier form of the risk track the regressor uses — the same `calibrate_risk`, the same facts, the
same certificate field (`assurance = action_risk_control`).


In [3]:
from collections import Counter

@dsl
class BinaryRiskPolicy(DecisionPolicyBase):
    assert rule("release_low_risk", when=lambda risk_release: risk_release,
                verdict=RELEASE)
    assert rule("deny_high_risk", when=lambda risk_deny: risk_deny,
                verdict=DENY)
    assert rule("review_middle",
                when=lambda risk_release, risk_deny:
                not risk_release and not risk_deny, verdict=ASK)

dm_risk_b = (dm_b.with_policy(BinaryRiskPolicy)
             .calibrate_risk(cal_b, ycal, estimator=forest_b,
                             alpha_release=0.10, alpha_deny=0.10))

ds = [dm_risk_b.dispose(r) for r in te_b]
print("selective-risk verdicts:", dict(Counter(d.verdict for d in ds)))
print("assurance             :", dict(Counter(d.assurance for d in ds)))
print("lambda release/deny   :",
      round(dm_risk_b.risk.lambda_release("__all__"), 3), "/",
      round(dm_risk_b.risk.lambda_deny("__all__"), 3))
print()

print(report(dm_risk_b, te_b, title="Breast cancer — selective risk (CRC)",
             safe=[int(v) == 0 for v in yte], decidable=[True] * len(yte),
             key=b"demo-audit-key", sample=6))


selective-risk verdicts: {'deny': 96, 'release': 46, 'abstain': 1}
assurance             : {'action_risk_control': 142, 'declared': 1}
lambda release/deny   : 0.27 / 0.27



 Breast cancer — selective risk (CRC)
 mjolnir.ml.report/1 · 2026-10-25T17:39:57Z

CONTRACT (the problem)
----------------------
  policy        BinaryRiskPolicy  (sha256:af4a389533497a87db57c56285022d02be340ebfbf4611ac037962def9c7ed31)
  model         RandomForestClassifier  (sha256:03ae4b5b6409c5b0…)
  calibration   alpha=0.10  (sha256:9627abc1181d023e…)
  risk track    event=adverse  alpha_release=0.10 · alpha_deny=0.10  (sha256:1a26bdbe6171e6f7…)
  assurance     coverage=pointwise_coverage · risk=action_risk_control · both=dual_track
  regime        required  method=mahalanobis  alpha_ood=0.02  tau=8241.46  (sha256:c367cc692d08c9d8…)
  features      mean_radius, mean_texture, mean_perimeter, mean_area, mean_smoothness, mean_compactness, mean_concavity, mean_concave_points, mean_symmetry, mean_fractal_dimension, radius_error, texture_error, perimeter_error, area_error, smoothness_error, compactness_error, concavity_error, concave_points_error, symmetry_error, fractal_dimension_error

## Multiclass — wine (all 13 features)

For a multiclass classifier there is no `p_positive`; `conf` is the max probability and the **conformal set** is the
interesting object. Rules can use `pred` (the point label) and `certified` (set singleton). We add a
stated-fact rule on `alcohol` to show feature facts next to model facts.

In [4]:
data = load_wine()
FEATURES_W = clean(data.feature_names)
Xtr, Xrest, ytr, yrest = train_test_split(data.data, data.target, test_size=0.5,
                                          stratify=data.target, random_state=0)
Xcal, Xte, ycal, yte = train_test_split(Xrest, yrest, test_size=0.5,
                                        stratify=yrest, random_state=0)
rows = lambda A: [dict(zip(FEATURES_W, map(float, x))) for x in A]
tr_w, cal_w, te_w = rows(Xtr), rows(Xcal), rows(Xte)

forest_w = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(Xtr, ytr)
print("features (all):", FEATURES_W)

@dsl
class WinePolicy(DecisionPolicyBase):
    assert rule("deny_high_alcohol",
                when=lambda alcohol: alcohol > 14.5, verdict=DENY)
    assert rule("deny_off_profile",
                when=lambda pred, conf: pred == 2 and conf >= 0.80, verdict=DENY)
    assert rule("release_approved",
                when=lambda certified, pred: certified and pred == 0, verdict=RELEASE)
    assert rule("review_unsettled",
                when=lambda certified: not certified, verdict=ASK)

dm_w = (DecisionModel(forest_w, WinePolicy, features=FEATURES_W)
        .fit_regime(tr_w, method="mahalanobis",
                    alpha_ood=0.05, calibration_rows=cal_w).calibrate(cal_w, ycal, alpha=0.10))

ledger_w = (HazardLedger()
            .add("high_alcohol", lambda r: r["alcohol"] > 13.5,
                 "product spec §1 (alcohol by volume)"))
print(report(dm_w, te_w, title="Wine — approve class 0, refuse the rest",
             safe=[int(v) == 0 for v in yte], decidable=[True] * len(yte),
             ledger=ledger_w, key=b"demo-audit-key", sample=8))

features (all): ('alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280_od315_of_diluted_wines', 'proline')


 Wine — approve class 0, refuse the rest
 mjolnir.ml.report/1 · 2026-10-25T17:39:59Z

CONTRACT (the problem)
----------------------
  policy        WinePolicy  (sha256:1260a379e70d68662f5f977f13b3f9b4247ce8e7c451d35e49490a57297d725a)
  model         RandomForestClassifier  (sha256:9cedeb416c1b59be…)
  calibration   alpha=0.10  (sha256:a80b0871cb5487df…)
  regime        required  method=mahalanobis  alpha_ood=0.05  tau=84.8591  (sha256:535546fb0c572536…)
  features      alcohol, malic_acid, ash, alcalinity_of_ash, magnesium, total_phenols, flavanoids, nonflavanoid_phenols, proanthocyanins, color_intensity, hue, od280_od315_of_diluted_wines, proline

POLICY (every statement, compiled)
----------------------------------
  code               kind     requires         predicate                  verdict
  -----------------  -------  ---------------  -------------------------  -------
  deny_high_alcohol  rule     alcohol          alcohol > 14.5             deny   
  deny_off_profile   rule  

### Replacing the 1D box with a multivariate regime floor

The axis-aligned box fails whenever **any one** feature steps outside its training min/max. In correlated
tabular data that rejects ordinary points: on the Wine split above, the 1D box rejects about a quarter of
the test rows even though they lie in the joint support. `fit_regime(..., method="mahalanobis")` fits a
robust center/covariance on the training rows and calibrates the anomaly cutoff on held-out rows, so the
**false-abstention rate is bounded at `alpha_ood`** (here 5%, because the calibration split has 44 rows).
`method="isolation_forest"` is the non-parametric alternative for multimodal or higher-dimensional data.

The calibrated cutoff, detector configuration and feature list are hashed into `regime_digest`, which
travels with every disposition and certificate.


In [5]:
import numpy as np
from collections import Counter

mins = {f: min(r[f] for r in tr_w) for f in FEATURES_W}
maxs = {f: max(r[f] for r in tr_w) for f in FEATURES_W}
box_pass = [all(mins[f] <= r[f] <= maxs[f] for f in FEATURES_W) for r in te_w]
floor_pass = [dm_w.regime_floor.check([r[f] for f in FEATURES_W])[0]
              for r in te_w]

print(f"1D box pass rate        : {np.mean(box_pass):.2%} "
      f"({sum(box_pass)}/{len(te_w)})")
print(f"multivariate floor pass : {np.mean(floor_pass):.2%} "
      f"({sum(floor_pass)}/{len(te_w)})")
print("floor tau / alpha_ood   :",
      round(dm_w.regime_floor.tau, 4), "/", dm_w.regime_floor.alpha_ood)
print("verdicts                :",
      dict(Counter(d.verdict for d in [dm_w.dispose(r) for r in te_w])))


1D box pass rate        : 57.78% (26/45)
multivariate floor pass : 93.33% (42/45)
floor tau / alpha_ood   : 84.8591 / 0.05


verdicts                : {'release': 13, 'abstain': 22, 'deny': 6, 'ask': 4}


## What to read in the report

- **DECISIONS** — how often the harness committed, escalated or refused; abstention causes are named
  (`out_of_regime`, `missing fact`, `uncovered`, premise codes).
- **OMEGA PAIR** — precision/recall of *claims* against the pinned oracle, with Wilson intervals, beside
  the degenerate ALWAYS/NEVER controls (NEVER has vacuous precision `⊥`, not 1).
- **HAZARDS** — completeness against the external ledger: a release on a hazard row is a false
  certificate. `observable recall` vs `hidden-only recall` separates declared coverage from luck.
- **CERTIFICATES** — signed and replayable counts; the auditor needs the policy artifact and the key,
  not the forest.

## Certificates — online replay, no forest, no training data

The certificate is the artifact: one row, checked **without the estimator**.
The published fact table carries the classifier's model facts (`pred`,
`conf`, `p_positive`, and — when the conformal or risk tracks are fitted —
`certified`, `set_size`, `p_event`, `risk_release`, `risk_deny`). Replay is
the same four-check path as in the regressor notebook — structure, HMAC
**signature**, id/row **binding**, and a fact-table **re-derivation**
through `Policy.dispose_facts`, the very function live `dispose()` ran — so
it is estimator-class-agnostic: the policy replays whatever facts the table
holds, whether they came from a regressor's `lo`/`hi` or a classifier's
`conf`/`certified`.

Harness-stage abstentions (`out_of_regime*`, `regime_not_fitted`, …) are
**signed and bound but attested, not replayed**: they depend on the training
bounds, which is why a report line reads `replayable from facts alone
87/89` when a couple of rows never reached the policy.

In [6]:
d0 = dm_w.dispose(te_w[0])
cert = certify(d0, key=b"demo-audit-key")
print("certificate id :", cert["certificate_id"][:34], "…")
print("verdict        :", cert["verdict"], "| basis:", cert["basis"])
print("facts          :", list(cert["facts"])[:6], "…")
print("verified       :", verify_certificate(cert, key=b"demo-audit-key",
                                            policy=WinePolicy, row=te_w[0]))
print("replayable     :", cert["replayable"])

# 1) signed verdict tamper -> the HMAC rejects it
bad = dict(cert, verdict="release" if cert["verdict"] != "release" else "deny")
try:
    verify_certificate(bad, key=b"demo-audit-key", policy=WinePolicy)
except Exception as ex:
    print("tampered verdict:", type(ex).__name__, "—", ex)

# 2) unsigned FACT tamper -> replay re-derives the verdict from the facts:
#    delete the fact the deciding rule requires -> abstain(missing) != verdict
need = dm_w.policy.statement(cert["basis"][0]).requires[0]
bad = dict(certify(d0),
           facts={k: v for k, v in cert["facts"].items() if k != need})
try:
    verify_certificate(bad, policy=WinePolicy)
except Exception as ex:
    print("tampered facts  :", type(ex).__name__, "—", ex)

# 3) rewritten certificate id -> the binding check recomputes it
from mjolnir_ml.certificate import _binding_id
bad = dict(certify(d0), certificate_id="sha256:" + "0" * 64)
try:
    verify_certificate(bad, policy=WinePolicy)
except Exception as ex:
    print("rewritten id    :", type(ex).__name__, "—", ex)

# 4) another row's inputs -> claim_digest mismatch
try:
    verify_certificate(cert, key=b"demo-audit-key", policy=WinePolicy,
                       row=te_w[1])
except Exception as ex:
    print("wrong row       :", type(ex).__name__, "—", ex)

certificate id : sha256:071719de12504dc430a48495d9f …
verdict        : release | basis: ['release_approved']
facts          : ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols'] …
verified       : True
replayable     : True
tampered verdict: CertificateError — signature does not match the certificate
tampered facts  : CertificateError — replay disagrees: certificate says (release, ['release_approved'], None), policy over the published facts says (abstain, ['missing:certified'], certified)
rewritten id    : CertificateError — certificate_id does not match the pinned digests (binding check failed)
wrong row       : CertificateError — input tamper: the supplied row does not hash to the certificate's claim_digest


## Takeaways

- The same harness handles binary and multiclass classifiers; only the policy changes.
- `certified` is a finite-sample guarantee (a conformal singleton), not a softmax threshold — that is
  what makes a release defensible.
- Stated-fact rules (`worst_area`, `alcohol`) sit beside model facts (`p_positive`, `conf`, `certified`) in
  one declared, digested table; `report()` shows both.
- `certified` (coverage) and `risk_release`/`risk_deny` (selective risk) are different guarantees; the
  same `calibrate_risk` call applies to classifiers, with the event being the dangerous class or an error.
- The multivariate regime floor (`mahalanobis` / `isolation_forest`) replaces the brittle 1D box and pins
  its calibrated cutoff in the certificate; the abstention rate is a declared `alpha_ood`, not a surprise.
- Swap RandomForest for LogisticRegression / HistGradientBoosting: the policy, κ and certificates are
  unchanged; only `model_digest` moves.